In [1]:
import numpy as np
import torch
from torchinfo import summary
import torch.nn as nn

from fileformer import FileFormer

In [2]:
model_f32 = FileFormer(257, 256, 4, 4, 0.0)
model_bf16 = FileFormer(257, 256, 4, 4, 0.0).to(torch.bfloat16)

In [3]:
summary(model_f32)

Layer (type:depth-idx)                   Param #
FileFormer                               --
├─Embedding: 1-1                         65,792
├─RotaryPositionalEmbeddings: 1-2        --
├─ModuleList: 1-3                        --
│    └─FileFormerBlock: 2-1              --
│    │    └─MultiHeadAttention: 3-1      262,400
│    │    └─FeedForward: 3-2             525,568
│    │    └─LayerNorm: 3-3               512
│    │    └─LayerNorm: 3-4               512
│    │    └─Dropout: 3-5                 --
│    └─FileFormerBlock: 2-2              --
│    │    └─MultiHeadAttention: 3-6      262,400
│    │    └─FeedForward: 3-7             525,568
│    │    └─LayerNorm: 3-8               512
│    │    └─LayerNorm: 3-9               512
│    │    └─Dropout: 3-10                --
│    └─FileFormerBlock: 2-3              --
│    │    └─MultiHeadAttention: 3-11     262,400
│    │    └─FeedForward: 3-12            525,568
│    │    └─LayerNorm: 3-13              512
│    │    └─LayerNorm: 3-14     

In [4]:
def model_memory_mb(model):

    total = 0
    for p in model.parameters():
        total += p.nelement() * p.element_size()
    return total / 1024 / 1024

In [5]:
print(model_memory_mb(model_bf16))
print(model_memory_mb(model_f32))

6.270998001098633
12.541996002197266


In [6]:
x = torch.randint(1, 200, (2, 1024), dtype=torch.long)

In [15]:
import time
import statistics

def benchmark(model, inp, steps=10, warmup=10, use_cuda=False):
    model.eval()
    with torch.no_grad():
        for _ in range(warmup):
            _ = model(inp, torch.zeros(x.shape, dtype=torch.bool))
        if use_cuda:
            torch.cuda.synchronize()
        times = []
        for _ in range(steps):
            t0 = time.perf_counter()
            _ = model(inp, torch.zeros(x.shape, dtype=torch.bool))
            if use_cuda:
                torch.cuda.synchronize()
            times.append((time.perf_counter() - t0) * 1000)
    return {
        "avg_ms": statistics.mean(times),
        "min_ms": min(times),
        "max_ms": max(times),
        "std_ms": statistics.pstdev(times),
        "fps": 1000.0 / statistics.mean(times),
    }


In [16]:
benchmark(model_f32, x)

{'avg_ms': 55.79633329762146,
 'min_ms': 52.79270900064148,
 'max_ms': 63.06520799989812,
 'std_ms': 2.8450185135621644,
 'fps': 17.922324656459622}

In [17]:
benchmark(model_bf16, x)

KeyboardInterrupt: 

In [4]:
model_f32(x, torch.zeros(x.shape, dtype=torch.bool))

tensor([[[-0.4353,  0.0876,  0.6858,  ..., -0.3332,  0.4694,  0.0759],
         [ 1.1871, -0.2936,  0.6666,  ...,  0.9443,  1.0498,  0.4150],
         [ 0.1850, -1.2321,  0.3738,  ..., -0.5865,  0.5846,  0.7131],
         ...,
         [ 0.3731,  0.6141,  0.2350,  ...,  0.2720, -0.4994, -0.1086],
         [-0.2565, -0.1693, -0.3521,  ..., -0.8639, -0.9070,  0.0924],
         [ 0.2606,  0.2050,  0.7065,  ..., -1.6608,  1.1806, -0.0259]],

        [[ 0.9754, -0.2958,  1.2977,  ..., -0.3082,  0.0092, -1.2975],
         [ 1.6198, -0.4301,  0.8265,  ..., -0.3559,  1.0872, -0.7506],
         [-1.0402, -0.3223,  1.6333,  ...,  0.2555,  0.4599, -0.3814],
         ...,
         [ 0.2128, -0.6704,  0.0830,  ..., -0.2113, -1.0066, -1.1726],
         [-0.3124, -0.0925,  0.8336,  ...,  0.3455,  0.2374,  0.0863],
         [ 0.4342, -0.0018,  0.5227,  ..., -0.6693, -0.0886,  0.6050]]],
       grad_fn=<ViewBackward0>)